In [1]:
import pandas as pd
import ast
import os
import glob

# 같은 폴더 내의 *_data_all.csv 파일을 모두 찾음
for file_path in glob.glob("./*_data_all.csv"):
    # 파일명 추출
    base_name = os.path.basename(file_path)
    # 예: cpu_data_all.csv → cpu.csv
    output_name = base_name.replace("_data_all.csv", ".csv")

    print(f"▶ {base_name} 정제 중...")

    # CSV 읽기
    df = pd.read_csv(file_path)

    # '상세정보' 컬럼 확인
    if "상세정보" not in df.columns:
        print(f"⚠️ {base_name}에는 '상세정보' 컬럼이 없습니다. 건너뜀.")
        continue

    # 문자열 형태의 딕셔너리를 실제 dict로 변환
    def parse_dict(x):
        try:
            return ast.literal_eval(str(x))
        except Exception:
            return {}

    df["상세정보"] = df["상세정보"].apply(parse_dict)

    # 상세정보를 컬럼으로 확장
    detail_df = pd.json_normalize(df["상세정보"])

    # 기존 데이터와 병합
    merged_df = pd.concat([df.drop(columns=["상세정보"]), detail_df], axis=1)

    # 결과 저장 (cpu.csv, gpu.csv, ...)
    merged_df.to_csv(output_name, index=False, encoding="utf-8-sig")

    print(f"✅ {output_name} 저장 완료!")

print("\n🎉 모든 파일 정제 완료!")


▶ case_data_all.csv 정제 중...
✅ case.csv 저장 완료!
▶ power_data_all.csv 정제 중...
✅ power.csv 저장 완료!

🎉 모든 파일 정제 완료!


In [ ]:
import pandas as pd 

df = pd.read_csv(r'C:\Users\82107\Desktop\4일 프로젝트 코드\pc_assembly\가공데이터\cpu_llm_amd.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 117 entries, 0 to 116
Data columns (total 33 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   제품명               117 non-null    object 
 1   가격                117 non-null    object 
 2   제조사               117 non-null    object 
 3   소켓 구분             117 non-null    object 
 4   코어 수              117 non-null    object 
 5   스레드 수             117 non-null    object 
 6   메모리 규격            117 non-null    object 
 7   내장그래픽             117 non-null    object 
 8   출시일               117 non-null    object 
 9   제조 공정             117 non-null    object 
 10  기본 클럭             117 non-null    object 
 11  최대 클럭             117 non-null    object 
 12  L2 캐시             117 non-null    object 
 13  L3 캐시             117 non-null    object 
 14  PCIe 버전           117 non-null    object 
 15  최대 PCIe 레인수       117 non-null    object 
 16  최대 메모리 크기         117 non-null    object 
 1

: 

In [ ]:
df = df.drop(columns=['Unnamed: 21','벤치마크','PPT','VR Ready 프리미엄','StoreMI','SenseMI','인텔 퀵싱크','그래픽 사양','KC인증','구성','사양','기술 지원'],axis=1)


In [8]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 546 entries, 0 to 545
Data columns (total 53 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   제품명               546 non-null    object 
 1   가격                546 non-null    object 
 2   제조사               546 non-null    object 
 3   인텔 CPU종류          416 non-null    object 
 4   소켓 구분             546 non-null    object 
 5   코어 수              546 non-null    object 
 6   스레드 수             546 non-null    object 
 7   메모리 규격            524 non-null    object 
 8   내장그래픽             538 non-null    object 
 9   출시일               370 non-null    object 
 10  제조 공정             528 non-null    object 
 11  사양                546 non-null    object 
 12  기본 클럭             546 non-null    object 
 13  최대 클럭             434 non-null    object 
 14  L2 캐시             334 non-null    object 
 15  L3 캐시             521 non-null    object 
 16  연산 체계             491 non-null    object 
 1

In [ ]:
df['제조사'].unique()

In [ ]:
df['제조사'] = df['제조사'].str.replace(r'AMD.*','AMD',regex=True)
df['제조사'] = df['제조사'].str.replace(r'인텔.*','INTEL',regex=True)


In [ ]:
amd_cpu = df['제조사'] == "AMD"
intel_cpu = df['제조사'] == 'INTEL'

amd = df[amd_cpu].copy()
intel = df[intel_cpu].copy()


In [ ]:
amd = amd.drop(columns=['인텔 CPU종류','버스 속도','PBP-MTP','인텔 XTU','인텔 딥러닝부스트','옵테인','메모리 사양','연산 체계'],axis=1)
intel = intel.drop(columns=['AMD CPU종류','세대 구분','AMD Ryzen Master','AMD 3D V캐시','AMD Ryzen AI','AMD PRO','메모리 사양'],axis=1)

In [ ]:
#패키지 형태가 중고인 부품은 제거 
amd = amd[amd['패키지 형태'] != '중고']
intel = intel[intel['패키지 형태'] != '중고']

In [ ]:
#LLM용 csv만들기 
amd = amd.fillna('정보없음')
intel = intel.fillna('정보없음')
amd.to_csv('cpu_llm_amd.csv',index=False)
intel.to_csv('cpu_llm_intel.csv',index=False)

In [ ]:
#원핫인코딩후 0/1로변환
amd = pd.get_dummies(amd,columns=['SMT(하이퍼스레딩)','AMD Ryzen Master','AMD 3D V캐시','AMD Ryzen AI','AMD PRO'])
intel = pd.get_dummies(intel,columns=['인텔 XTU','인텔 딥러닝부스트','SMT(하이퍼스레딩)','옵테인'])
bool_cols = amd.select_dtypes(include='bool').columns
amd[bool_cols] = amd[bool_cols].astype('uint8')
intel_bool_cols = intel.select_dtypes(include='bool').columns
intel[intel_bool_cols] = intel[intel_bool_cols].astype('uint8')


In [ ]:
intel['메모리 채널'].unique()

In [ ]:
intel.info()

In [ ]:
#intel 데이터 숫자형으로 변환
import numpy as np
import re 
# intel['P_코어'] = intel['코어 수'].astype(str).str.extract(r'(\d+)(?=C|\()|^(?!.*[+]).*?(\d+)', expand=True).astype(float).fillna(0).max(axis=1).astype('Int64')
# intel['E_코어'] = intel['코어 수'].astype(str).str.extract(r'\+(\d+)c', expand=False).astype(float).fillna(0).astype('Int64')
# intel = intel.drop(columns=['코어 수'])
intel['스레드 수'] = intel['스레드 수'].astype(str).str.replace(r'[^\d.]', '', regex=True).astype(int)
intel['TDP'] = intel['TDP'].astype(str).str.replace(r'[^\d.]','',regex=True).replace('',np.nan).fillna('0').astype(int)
intel['GPU 코어 속도'] = intel['GPU 코어 속도'].astype(str).str.replace({'MHz',','},'').fillna('0').astype(int)
intel['기본 클럭'] = intel['기본 클럭'].astype(str).str.replace('GHz','').fillna('0').astype(float)
intel['최대 클럭'] = intel['최대 클럭'].astype(str).str.findall(r'\d+\.?\d*').apply(lambda x: float(x[-1]) if x else 0.0).astype(float)
intel['L3 캐시'] = intel['L3 캐시'].astype(str).str.replace('MB','').fillna('0').astype(float)
intel['L2 캐시'] = (intel['L2 캐시'].astype(str).str.upper().str.replace(' ', '').apply(lambda s: 
           float(re.search(r'(\d+)KBX(\d+)', s).group(1)) * float(re.search(r'(\d+)KBX(\d+)', s).group(2)) / 1024.0 if 'KBX' in s else
           float(re.search(r'(\d+)', s).group(1)) / 1024.0 if 'KB' in s else
           float(re.search(r'(\d+)MBX(\d+)', s).group(1)) * float(re.search(r'(\d+)MBX(\d+)', s).group(2)) if 'MBX' in s else
           float(re.search(r'(\d+\.?\d*)', s).group(1)) if 'MB' in s else
           0.0)
    .astype(float)
)
intel['PCIe 버전'] = (
    intel['PCIe 버전'].astype(str)
    .str.findall(r'(\d+\.?\d*)') 
    .apply(lambda x: float(max(x)) if x else 0.0)
    .astype(float) # float으로 통일
)
intel['최대 PCIe 레인수'] = intel['최대 PCIe 레인수'].astype(str).str.replace(r'\s*\([^)]*\)', '', regex=True).str.extract(r'(\d+)', expand=False).fillna('0').astype(int)
intel['최대 메모리 크기'] = (
    intel['최대 메모리 크기'].astype(str).str.upper()
    .apply(lambda s: 
           float(re.search(r'(\d+\.?\d*)', s).group(1)) * 1024.0 if 'TB' in s else
           float(re.search(r'(\d+\.?\d*)', s).group(1)) if 'GB' in s else
           0.0)
    .astype(float)
)
df['메모리 클럭'] = (
    df['메모리 클럭'].astype(str)
    .str.findall(r'(\d+\.?\d*)') 
    .apply(lambda x: float(max(x)) if x else 0.0)
    .astype(float)
)
intel['메모리 채널'] = intel['메모리 채널'].astype(str).str.replace(r'[^\d.]','',regex=True).replace('',0).astype(float)


In [ ]:
intel.to_csv('cpu_imbedded_intel.csv',index=False)

In [ ]:
#amd 데이터 숫자형으로 변환 
amd['P_코어'] = amd['코어 수'].astype(str).str.extract(r'(\d+)(?=C|\()|^(?!.*[+]).*?(\d+)', expand=True).astype(float).fillna(0).max(axis=1).astype('Int64')
amd['E_코어'] = amd['코어 수'].astype(str).str.extract(r'\+(\d+)c', expand=False).astype(float).fillna(0).astype('Int64')
amd = amd.drop(columns=['코어 수'])
amd['스레드 수'] = amd['스레드 수'].astype(str).str.replace('스레드', '', regex=True).astype(int)
amd['TDP'] = amd['TDP'].astype(str).str.replace('W','').fillna('0').astype(int)
amd['GPU 코어 속도'] = amd['GPU 코어 속도'].astype(str).str.replace({'MHz',','},'').fillna('0').astype(int)
amd['기본 클럭'] = amd['기본 클럭'].astype(str).str.replace('GHz','').fillna('0').astype(float)
amd['최대 클럭'] = amd['최대 클럭'].astype(str).str.replace('GHz','').fillna('0').astype(float)
amd['L3 캐시'] = amd['L3 캐시'].astype(str).str.replace('MB','').fillna('0').astype(int)
amd['L2 캐시'] = amd['L2 캐시'].astype(str).str.replace('MB','').fillna('0').astype(int)
amd['PCIe 버전'] = amd['PCIe 버전'].astype(str).str.replace('PCIe','').fillna('0').astype(float)
amd['최대 PCIe 레인수'] = amd['최대 PCIe 레인수'].astype(str).str.replace(r'\s*\([^)]*\)', '', regex=True).str.extract(r'(\d+)', expand=False).fillna('0').astype(int)
amd['최대 메모리 크기'] = amd['최대 메모리 크기'].astype(str).str.replace(r'[^\d.]','',regex=True).replace('',0).astype(int)
amd['메모리 클럭'] = amd['메모리 클럭'].astype(str).str.replace('MHz','').fillna('0').astype(float)
amd['메모리 채널'] = amd['메모리 채널'].astype(str).str.replace(r'[^\d.]','',regex=True).replace('',0).astype(float)


In [ ]:
amd.to_csv('cpu_imbedded_amd.csv',index=False)